In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from collections import defaultdict

# 数据准备
data = [
    {
        "text": "arrive Shanghai on November 2nd and",
        "entities": [
            {"start": 7, "end": 15, "label": "DES"},  # Shanghai
            {"start": 19, "end": 31, "label": "TIM"}  # November 2nd
        ]
    },
    {
        "text": "leave Shanghai on November 2nd",
        "entities": [
            {"start": 6, "end": 14, "label": "DEP"},  # Shanghai
            {"start": 18, "end": 30, "label": "TIM"}  # November 2nd
        ]
    }
]
data0 = [
    {
        "text": "John lives in New York",
        "entities": [
            {"start": 0, "end": 4, "label": "PER"},  # John
            {"start": 13, "end": 22, "label": "LOC"}  # New York
        ]
    },
    {
        "text": "Alice works at Google",
        "entities": [
            {"start": 0, "end": 5, "label": "PER"},  # Alice
            {"start": 15, "end": 21, "label": "ORG"}  # Google
        ]
    }
]

# 构建词汇表和标签表
word_counter = defaultdict(int)
label_set = set()

for example in data:
    for word in example["text"].split():
        word_counter[word] += 1
    for entity in example["entities"]:
        label_set.add(entity["label"])

vocab = {word: idx + 1 for idx, word in enumerate(word_counter)}  # 索引从 1 开始
vocab["<pad>"] = 0  # 填充符

label_to_id = {label: idx+2 for idx, label in enumerate(label_set)}
label_to_id["O"] = 1
label_to_id["<pad>"] = 0  # 填充符
id_to_label = {idx: label for label, idx in label_to_id.items()}

print(label_to_id)
print(id_to_label)
print(vocab)

{'TIM': 2, 'DES': 3, 'DEP': 4, 'O': 1, '<pad>': 0}
{2: 'TIM', 3: 'DES', 4: 'DEP', 1: 'O', 0: '<pad>'}
{'arrive': 1, 'Shanghai': 2, 'on': 3, 'November': 4, '2nd': 5, 'and': 6, 'leave': 7, '<pad>': 0}


In [28]:

# 数据预处理
def prepare_example(example, vocab, label_to_id):
    text = example["text"]
    entities = example["entities"]
    
    words = text.split()
    word_indices = [vocab[word] for word in words]
    #print(word_indices)
    labels = ["O"] * len(words)
    for entity in entities:
        start = entity["start"]
        end = entity["end"]
        label = entity["label"]
        entity_words = text[start:end].split()
        print(entity_words)
        for i in range(len(words)):
            if words[i] in entity_words:
                labels[i] = label
    #print(labels)
    label_indices = [label_to_id[label] for label in labels]
    #print(label_indices)
    return word_indices, label_indices

train_data = [prepare_example(example, vocab, label_to_id) for example in data]
print(train_data)

['Shanghai']
['November', '2nd']
['Shanghai']
['November', '2nd']
[([1, 2, 3, 4, 5, 6], [1, 3, 1, 2, 2, 1]), ([7, 2, 3, 4, 5], [1, 4, 1, 2, 2])]


In [29]:

# 定义模型
class NERModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.RNN(embedding_dim, hidden_dim,num_layers=1, 
                            bidirectional=False, batch_first=False)
        self.fc = nn.Linear(hidden_dim , output_dim)
    
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        logits = self.fc(lstm_out)
        return logits


In [30]:

from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    
    # 填充批次中的句子
    src_batch = pad_sequence([torch.tensor(s) for s in src_batch], padding_value=0)
    trg_batch = pad_sequence([torch.tensor(t) for t in trg_batch], padding_value=0)
    
    return src_batch, trg_batch

# 准备数据加载器
class NERDataset(Dataset):
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

dataset = NERDataset(train_data)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)


In [33]:

# 初始化模型和优化器
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 16
HIDDEN_DIM = 16
OUTPUT_DIM = len(label_to_id)

model = NERModel(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM)
criterion = nn.CrossEntropyLoss(ignore_index=0)
#optimizer = optim.Adam(model.parameters(),lr=0.001)
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# 训练模型
NUM_EPOCHS = 100

for epoch in range(NUM_EPOCHS):
    for word_indices, label_indices in dataloader:
        word_tensor = torch.tensor(word_indices, dtype=torch.long)
        label_tensor = torch.tensor(label_indices, dtype=torch.long)
        #print("word_tensor",word_tensor)
        #print("label_tensor",label_tensor)
        logits = model(word_tensor)
        #print("word_tensor.shape:",word_tensor.shape)
        loss = criterion(logits.view(-1, OUTPUT_DIM), label_tensor.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if epoch % 10 == 9:
        print(f"Epoch {epoch+1}, Loss: {loss.item()}")


Epoch 10, Loss: 0.7267463803291321
Epoch 20, Loss: 0.2914603054523468
Epoch 30, Loss: 0.166093111038208
Epoch 40, Loss: 0.1391759216785431
Epoch 50, Loss: 0.14151306450366974
Epoch 60, Loss: 0.11774426698684692
Epoch 70, Loss: 0.10262610018253326
Epoch 80, Loss: 0.0916602611541748
Epoch 90, Loss: 0.06859616190195084
Epoch 100, Loss: 0.04813019186258316


/var/folders/bz/tdtgpb8s1p798947fmhm_mzm0000gn/T/ipykernel_25044/463006727.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  word_tensor = torch.tensor(word_indices, dtype=torch.long)
/var/folders/bz/tdtgpb8s1p798947fmhm_mzm0000gn/T/ipykernel_25044/463006727.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  label_tensor = torch.tensor(label_indices, dtype=torch.long)


In [34]:

# 测试模型
def predict(model, sentence, vocab, id_to_label):
    model.eval()
    words = sentence.split()
    word_indices = [vocab.get(word, 0) for word in words]
    word_tensor = torch.tensor([word_indices], dtype=torch.long)
    word_tensor = word_tensor.transpose(1,0)# 转成（sentence_len, batch_size）
    
    with torch.no_grad():
        #print("word:",word_tensor.flatten())
        #print("word_tensor.shape:",word_tensor.shape)
        logits = model(word_tensor)
        predictions = torch.argmax(logits, dim=-1)
        #print("pre:",predictions.flatten())
        #print("logits:",logits)
    labels = [id_to_label[idx.item()] for idx in predictions]
    return list(zip(words, labels))

#sentences = ["John lives in Google","New York lives in John"]
sentences =[ "leave Shanghai on November 2nd", "arrive Shanghai on November 2nd"]
for sentence in sentences:
    predictions = predict(model, sentence, vocab, id_to_label)
    print(predictions)

[('leave', 'O'), ('Shanghai', 'DEP'), ('on', 'O'), ('November', 'TIM'), ('2nd', 'TIM')]
[('arrive', 'O'), ('Shanghai', 'DES'), ('on', 'O'), ('November', 'TIM'), ('2nd', 'TIM')]
